# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad-ahmed-developer/flyRank_Internship_Tasks/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

My lane: **Refresh / Content Opportunity Scoring**

**Question:** Can the observable characteristics and performance signals of pages help us prioritize which pages deserve human review?

I am choosing the **Refresh / Content Opportunity Scoring** lane because I want to investigate whether the available search, content, and engagement data can help identify pages that may deserve attention. The goal is to explore whether these signals can be used to rank pages by their review priority instead of treating every page equally. I chose this lane because the starter dataset contains relevant information such as impressions, clicks, sessions, content age, freshness, position, CTR, engagement, and trends.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # Walk up the directory tree until data/raw is found
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")



Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [15]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

lane_columns = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "trend_direction",
    "trend_pct"
]

print("\nRelevant columns available:")
print(df[lane_columns].columns.tolist())

Dataset shape: (30000, 44)

Relevant columns available:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'trend_direction', 'trend_pct']


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

The decision I want to improve is **which pages should be reviewed first** when the SEO team has limited time and resources. The SEO experts would use the ranked results to investigate the selected pages and decide whether they should be refreshed, expanded, protected, pruned, or monitored. A wrong recommendation could waste time and content resources on a low-priority page, while failing to recommend an important page could cause a useful review opportunity to be missed.


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

total_pages = df["content_id"].nunique()
total_clients = df["client_id"].nunique()

pages_per_client = df.groupby("client_id")["content_id"].nunique()

print(f"Unique pages: {total_pages:,}")
print(f"Unique clients: {total_clients:,}")
print(f"Average pages per client: {pages_per_client.mean():.1f}")
print(f"Maximum pages for one client: {pages_per_client.max():,}")

Unique pages: 30,000
Unique clients: 32
Average pages per client: 937.5
Maximum pages for one client: 7,008


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

The starter data contains **30,000 unique pages**, of which **54.21% show a downward trend**. In addition, **55.75% pages have at least 500 impressions over the 90-day period**. These numbers show that the dataset contains a substantial number of pages with measurable search visibility and different performance trends, making it reasonable to investigate whether pages can be prioritized for review.


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

total_pages = df["content_id"].nunique()
print(f"Total unique pages: {total_pages:,}")

declining_pages = (df["trend_direction"] == "down").sum()
declining_pct = (df["trend_direction"] == "down").mean() * 100
print(f"\nPages with downward trend: {declining_pages:,}")
print(f"Percentage with downward trend: {declining_pct:.2f}%")

visible_pages = (df["impressions_90d"] >= 500).sum()
visible_pct = (df["impressions_90d"] >= 500).mean() * 100
print(f"\nPages with at least 500 impressions: {visible_pages:,}")
print(f"Percentage with at least 500 impressions: {visible_pct:.2f}%")

Total unique pages: 30,000

Pages with downward trend: 16,262
Percentage with downward trend: 54.21%

Pages with at least 500 impressions: 16,726
Percentage with at least 500 impressions: 55.75%


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

My work will provide **observed, directional, and decision-support information** about which pages may deserve review based on measurable data. I can evaluate whether the available signals are useful for prioritizing pages under the chosen evaluation method. However, the model will not guarantee that a recommended page is the correct page to update, prove that updating a page will improve its performance, or explain Google's ranking algorithm. I also cannot claim that a relationship in the data is a causal effect without additional evidence.


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Trend direction values:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nColumns containing action-related terms:")
action_columns = []

for col in df.columns:
    col_lower = col.lower()
    for word in ["action", "refresh", "update"]:
        if word in col_lower:
            action_columns.append(col)
            break

print(action_columns)

Trend direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Columns containing action-related terms:
['days_since_last_update']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.